# Credit Card Fraud Detection

**Dataset:** Credit Card Fraud Detection (Kaggle — `mlg-ulb/creditcardfraud`).

It contains transactions made by European cardholders in September 2013.
- 284,807 transactions, 492 of which are frauds (~0.172%).
- Features `V1`–`V28` are PCA-transformed (anonymised for privacy).
- `Time` and `Amount` are the only non-PCA features.
- `Class` is the target: 1 = fraud, 0 = legitimate.

**Source:** https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud

## 1. Download the dataset

Run this once. Requires a Kaggle API token (`~/.kaggle/kaggle.json`).
If you already have `creditcard.csv` locally, skip this cell.

In [ ]:
# !pip install kaggle pandas numpy matplotlib seaborn scikit-learn imbalanced-learn -q
# !kaggle datasets download -d mlg-ulb/creditcardfraud --unzip

## 2. Imports & load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, precision_recall_curve,
                             average_precision_score, accuracy_score)

sns.set(style='whitegrid')
df = pd.read_csv('archive/creditcard.csv')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:', df.isnull().sum().sum())
df.info()

In [ ]:
df.describe()

### Q1 — % of fraud transactions in the dataset

In [ ]:
total = len(df)
fraud_count = (df['Class'] == 1).sum()
legit_count = (df['Class'] == 0).sum()
fraud_pct = 100 * fraud_count / total

print(f'Total transactions : {total}')
print(f'Legit transactions : {legit_count}')
print(f'Fraud transactions : {fraud_count}')
print(f'% fraud            : {fraud_pct:.4f}%')

### Q2 — Average transaction amount for fraud transactions

In [ ]:
avg_fraud_amount = df.loc[df['Class'] == 1, 'Amount'].mean()
avg_legit_amount = df.loc[df['Class'] == 0, 'Amount'].mean()

print(f'Average fraud amount  : {avg_fraud_amount:.2f}')
print(f'Average legit amount  : {avg_legit_amount:.2f}  (for comparison)')

# Side-by-side summary
df.groupby('Class')['Amount'].agg(['mean', 'median', 'min', 'max', 'count'])

In [ ]:
# Class distribution
counts = df['Class'].value_counts()
print(counts)
print('\nFraud ratio: {:.4f}%'.format(100 * counts[1] / counts.sum()))

sns.countplot(x='Class', data=df)
plt.title('Class distribution (0 = legit, 1 = fraud)')
plt.show()

In [ ]:
# Amount and Time distributions for fraud vs legit
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[df.Class == 0]['Amount'], bins=50, ax=ax[0], color='blue', label='Legit')
sns.histplot(df[df.Class == 1]['Amount'], bins=50, ax=ax[0], color='red', label='Fraud')
ax[0].set_yscale('log'); ax[0].set_title('Amount distribution'); ax[0].legend()

sns.histplot(df[df.Class == 0]['Time'], bins=50, ax=ax[1], color='blue', label='Legit')
sns.histplot(df[df.Class == 1]['Time'], bins=50, ax=ax[1], color='red', label='Fraud')
ax[1].set_title('Time distribution'); ax[1].legend()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(), cmap='coolwarm', center=0)
plt.title('Feature correlation')
plt.show()

## 4. Is there class imbalance? What can we do?

**Yes — extreme class imbalance.** Frauds make up only ~0.172% of transactions (492 out of 284,807). The model can trivially predict "not fraud" for everything and still look excellent on accuracy.

**What we can do:**
1. **Resampling**
   - *Oversampling* the minority class (e.g., **SMOTE** — synthetic minority oversampling).
   - *Undersampling* the majority class (random undersampling).
2. **Class weighting** — set `class_weight='balanced'` so misclassifying a fraud costs much more during training.
3. **Use the right metrics** — Precision, Recall, F1, ROC-AUC, and especially **PR-AUC** (Average Precision), which is more informative under heavy imbalance.
4. **Threshold tuning** — instead of the default 0.5, pick a threshold from the precision-recall curve that matches the business cost of false positives vs missed frauds.
5. **Anomaly detection** approaches (Isolation Forest, One-Class SVM, autoencoders) when frauds are essentially outliers.

## 5. Is accuracy a good measure here?

**No.** Because 99.83% of transactions are legitimate, a dummy classifier that predicts "legit" for every transaction achieves **99.83% accuracy** while catching **zero frauds** — which is worthless in production. Accuracy is dominated by the majority class and hides the model's behaviour on the class we actually care about.

**Better metrics for this problem:**
- **Recall (Sensitivity)** — of all real frauds, how many did we catch? Missing a fraud is very expensive.
- **Precision** — of the alerts we raised, how many were really fraud? Too many false positives annoy customers and waste review time.
- **F1-score** — harmonic mean of precision and recall.
- **PR-AUC (Average Precision)** — preferred over ROC-AUC under heavy imbalance because ROC-AUC can look optimistic when negatives vastly outnumber positives.
- **Confusion matrix** — to inspect TP / FP / FN / TN directly.

## 6. ML model — which one, and why?

**Choice: Logistic Regression with `class_weight='balanced'`.**

Reasons:
1. The features `V1`–`V28` are already PCA components — linear, decorrelated, and standardised. A linear model is a strong, natural fit.
2. It is **fast** to train on ~285k rows and **scales** well — important for production fraud screening where latency matters.
3. It is **interpretable** — coefficients show which components push a transaction toward fraud, useful for analyst review and audit.
4. Built-in `class_weight='balanced'` cleanly handles the imbalance without resampling tricks.
5. It outputs **calibrated probabilities**, so we can tune the decision threshold to match business cost.

(Tree-based models like Random Forest / XGBoost often score slightly higher PR-AUC on this dataset and are reasonable alternatives, but logistic regression is the right baseline given the PCA features.)

## 7. Train / test split & scaling

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# Stratify keeps the same fraud ratio in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale Time and Amount (V1-V28 are already PCA-scaled)
scaler = StandardScaler()
X_train[['Time', 'Amount']] = scaler.fit_transform(X_train[['Time', 'Amount']])
X_test[['Time', 'Amount']] = scaler.transform(X_test[['Time', 'Amount']])

print('Train:', X_train.shape, '  Frauds:', y_train.sum())
print('Test :', X_test.shape, '  Frauds:', y_test.sum())

## 8. Train the model

In [ ]:
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(X_train, y_train)

## 9. Evaluate

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('Accuracy :', accuracy_score(y_test, y_pred))
print('ROC-AUC  :', roc_auc_score(y_test, y_proba))
print('PR-AUC   :', average_precision_score(y_test, y_proba))
print('\nClassification report:')
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Precision-Recall curve — the right curve to look at under imbalance
prec, rec, _ = precision_recall_curve(y_test, y_proba)
plt.plot(rec, prec)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall curve (AP = {:.3f})'.format(
    average_precision_score(y_test, y_proba)))
plt.show()

## 10. Takeaways

- The data is **severely imbalanced** (~0.17% fraud). Naive models look great on accuracy but catch no fraud.
- **Accuracy is misleading**; use **Recall, Precision, F1, and PR-AUC** instead.
- **Logistic Regression with balanced class weights** is a strong, fast, interpretable baseline that fits the PCA-transformed features well.
- Further improvements: SMOTE / undersampling, threshold tuning on the PR curve, or gradient-boosted trees (XGBoost / LightGBM).